### Check received files

In [0]:
source_path = "/Volumes/customer360_dev/staging/source_files/customers/initial"

files = dbutils.fs.ls(source_path)

display(files)

path,name,size,modificationTime
dbfs:/Volumes/customer360_dev/staging/source_files/customers/initial/customers_initial_001.csv,customers_initial_001.csv,15921909,1790404144000
dbfs:/Volumes/customer360_dev/staging/source_files/customers/initial/customers_initial_002.csv,customers_initial_002.csv,16033022,1790404144000
dbfs:/Volumes/customer360_dev/staging/source_files/customers/initial/customers_initial_003.csv,customers_initial_003.csv,16033029,1790404144000
dbfs:/Volumes/customer360_dev/staging/source_files/customers/initial/customers_initial_004.csv,customers_initial_004.csv,16033017,1790404144000
dbfs:/Volumes/customer360_dev/staging/source_files/customers/initial/customers_initial_005.csv,customers_initial_005.csv,16033022,1790404144000
dbfs:/Volumes/customer360_dev/staging/source_files/customers/initial/customers_initial_006.csv,customers_initial_006.csv,16033029,1790404144000
dbfs:/Volumes/customer360_dev/staging/source_files/customers/initial/customers_initial_007.csv,customers_initial_007.csv,16033017,1790404143000
dbfs:/Volumes/customer360_dev/staging/source_files/customers/initial/customers_initial_008.csv,customers_initial_008.csv,16033022,1790404144000
dbfs:/Volumes/customer360_dev/staging/source_files/customers/initial/customers_initial_009.csv,customers_initial_009.csv,16033029,1790404144000
dbfs:/Volumes/customer360_dev/staging/source_files/customers/initial/customers_initial_010.csv,customers_initial_010.csv,16033017,1790404144000


In [0]:
csv_files = [
    f for f in files
    if f.name.endswith(".csv")
]

print("CSV files received:", len(csv_files))

for f in csv_files:
    print(f.name, f.size)

CSV files received: 10
customers_initial_001.csv 15921909
customers_initial_002.csv 16033022
customers_initial_003.csv 16033029
customers_initial_004.csv 16033017
customers_initial_005.csv 16033022
customers_initial_006.csv 16033029
customers_initial_007.csv 16033017
customers_initial_008.csv 16033022
customers_initial_009.csv 16033029
customers_initial_010.csv 16033017


### Read the files as source data

In [0]:
customers_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(source_path)
)

### Validate total row count

In [0]:
received_count = customers_df.count()

print(f"Rows received: {received_count:,}")

Rows received: 1,000,000


In [0]:
customers_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_number: long (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- gender: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- customer_status: string (nullable = true)
 |-- loyalty_tier: string (nullable = true)
 |-- preferred_language: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)



In [0]:
display(customers_df.limit(20))

customer_id,first_name,last_name,email,phone_number,date_of_birth,gender,signup_date,customer_status,loyalty_tier,preferred_language,created_at,updated_at
CUST000200001,Meera,Reddy,meera.reddy200001@example.com,917583807919,1978-10-17,MALE,2025-02-24,ACTIVE,SILVER,TAMIL,2025-02-24T09:00:00.000Z,2025-08-22T12:27:00.000Z
CUST000200002,Vikram,Kumar,vikram.kumar200002@example.com,917583815838,1978-11-23,OTHER,2025-03-13,INACTIVE,GOLD,HINDI,2025-03-13T09:00:00.000Z,2025-10-07T15:34:00.000Z
CUST000200003,Sanjay,Patel,sanjay.patel200003@example.com,917583823757,1978-12-30,FEMALE,2025-03-30,SUSPENDED,PLATINUM,KANNADA,2025-03-30T09:00:00.000Z,2025-11-22T18:41:00.000Z
CUST000200004,Rahul,Rao,rahul.rao200004@example.com,917583831676,1979-02-05,MALE,2025-04-16,ACTIVE,BRONZE,TELUGU,2025-04-16T09:00:00.000Z,2026-01-07T21:48:00.000Z
CUST000200005,Kiran,Singh,kiran.singh200005@example.com,917583839595,1979-03-14,OTHER,2025-05-03,ACTIVE,BRONZE,ENGLISH,2025-05-03T09:00:00.000Z,2026-02-23T00:55:00.000Z
CUST000200006,Ravi,Nair,ravi.nair200006@example.com,917583847514,1979-04-20,FEMALE,2025-05-20,ACTIVE,SILVER,TAMIL,2025-05-20T09:00:00.000Z,2026-04-10T03:02:00.000Z
CUST000200007,Kavya,Iyer,kavya.iyer200007@example.com,917583855433,1979-05-27,MALE,2025-06-06,ACTIVE,SILVER,HINDI,2025-06-06T09:00:00.000Z,2026-05-26T06:09:00.000Z
CUST000200008,Arjun,Gupta,arjun.gupta200008@example.com,917583863352,1979-07-03,OTHER,2025-06-23,INACTIVE,GOLD,KANNADA,2025-06-23T09:00:00.000Z,2025-07-10T09:16:00.000Z
CUST000200009,Divya,Verma,divya.verma200009@example.com,917583871271,1979-08-09,FEMALE,2025-07-10,SUSPENDED,PLATINUM,TELUGU,2025-07-10T09:00:00.000Z,2025-08-25T12:23:00.000Z
CUST000200010,Aarav,Sharma,aarav.sharma200010@example.com,917583879190,1979-09-15,MALE,2025-07-27,ACTIVE,BRONZE,ENGLISH,2025-07-27T09:00:00.000Z,2025-10-10T15:30:00.000Z


Check basic source issues

In [0]:
from pyspark.sql import functions as F

customers_df.select(
    F.count("*").alias("total_rows"),

    F.sum(
        F.when(
            F.col("customer_id").isNull() |
            (F.trim(F.col("customer_id")) == ""),
            1
        ).otherwise(0)
    ).alias("missing_customer_ids"),

    F.sum(
        F.when(
            ~F.col("email").contains("@"),
            1
        ).otherwise(0)
    ).alias("obviously_invalid_emails"),

    F.sum(
        F.when(
            ~F.col("customer_status").isin(
                "ACTIVE", "INACTIVE", "SUSPENDED"
            ),
            1
        ).otherwise(0)
    ).alias("invalid_statuses")

).show()

+----------+--------------------+------------------------+----------------+
|total_rows|missing_customer_ids|obviously_invalid_emails|invalid_statuses|
+----------+--------------------+------------------------+----------------+
|   1000000|                1000|                    5000|            2000|
+----------+--------------------+------------------------+----------------+

